In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.linalg import expm
from tqdm import tqdm

from kooplearn._linalg import eigh_rank_reveal, spd_neg_pow, weighted_norm
from kooplearn.datasets import compute_prinz_potential_eig, make_prinz_potential
from kooplearn.kernel import KernelRidge


def operator_norm_error(true_operator: np.ndarray, estimated_operator: np.ndarray):
    r"""Operator norm error proxy for a Koopman estimator.

    Computes the operator norm discrepancy between the true action
    :math:`A_\pi S` and the estimated action :math:`S \widehat{G}`:

    .. math::

        \mathcal{E}(\widehat{G}) := \|A_\pi S - S \widehat{G}\|.

    Since "kooplearn" does not currently expose :math:`A_\pi` or the embedding
    operator :math:`S` explicitly, this function works with their actions on a
    common finite-dimensional representation. In practice, the caller should pass
    matrices or vectors representing the two quantities to be compared.
    """
    true_operator = np.asanyarray(true_operator)
    estimated_operator = np.asanyarray(estimated_operator)

    if true_operator.shape != estimated_operator.shape:
        raise ValueError(
            "true_operator and estimated_operator must have the same "
            f"shape, got {true_operator.shape} and "
            f"{estimated_operator.shape}."
        )

    diff = true_operator - estimated_operator
    if diff.ndim == 1:
        return float(np.linalg.norm(diff))
    return float(np.linalg.norm(diff, ord=2))


def metric_distortion(psi, C):
    r"""Empirical metric distortion :math:`\widehat\eta_i = \|\widehat\psi_i\|_{\mathcal H} /
    \sqrt{\langle \widehat C \widehat\psi_i, \widehat\psi_i\rangle}`.

    Parameters
    ----------
    psi : ndarray, shape (n,) or (n, k)
        Eigenfunction(s) evaluated at the *training* points. If 2D, each
        column is treated as a separate eigenfunction (see `weighted_norm`).
    C : ndarray, shape (n, n)
        Empirical (kernel-based) covariance, i.e. ``model.kernel_X / n_samples``.
    """
    psi = np.asarray(psi)
    n = C.shape[0]

    # ||psi||_H via the reproducing property: needs the *inverse* Gram, since
    # C = K_X / n is the Gram-based covariance, not the RKHS metric itself.
    C_inv = spd_neg_pow(C * n, exponent=-1.0)  # i.e. K_X^{-1}
    rkhs_norm = weighted_norm(psi, M=C_inv)

    # <C psi, psi> = (1/n)||psi(X)||_2^2, i.e. weighted_norm with M=None, squared, over n
    empirical_norm = weighted_norm(psi, M=None) / np.sqrt(n)

    with np.errstate(divide="ignore", invalid="ignore"):
        eta = rkhs_norm / empirical_norm
    eta = np.where(empirical_norm > 0, eta, np.nan)
    return eta if psi.ndim == 2 else float(eta)


def spectral_bias(eigenfunction, C, rho):
    r"""Empirical spectral bias :math:`\hat s_i = \widehat\eta_i \, \rho_{r+1}`."""
    eta = metric_distortion(eigenfunction, C)
    s_hat = eta * rho
    return float(s_hat), eta


def _top_sv(C, r):
    """(r+1)-st eigenvalue of a symmetric PSD matrix, via eigh_rank_reveal."""
    raw_vals, raw_vecs = np.linalg.eigh(np.asarray(C))
    _, top_vals, _ = eigh_rank_reveal(raw_vals, raw_vecs, rank=r + 1)
    if len(top_vals) <= r:
        return 0.0
    return float(top_vals[-1])


# --- truncation helpers ---
def pcr_truncation(C, r):
    r""":math:`\rho_{r+1}(\widehat G^{PCR}) = \sigma_{r+1}(\widehat C)`."""
    return _top_sv(C, r)


# kDMD uses the same (r+1)-st eigenvalue of the empirical covariance as PCR
kdmd_truncation = pcr_truncation


def rrr_truncation(C, T, r, cutoff=None):
    r""":math:`\rho_{r+1}(\widehat G^{RRR}) = \sigma_{r+1}(\widehat C^{-1/2}\widehat T)`."""
    C_inv_sqrt = spd_neg_pow(np.asarray(C), exponent=-0.5, cutoff=cutoff)
    A = C_inv_sqrt @ np.asarray(T)
    svals = np.linalg.svd(A, compute_uv=False)
    if r >= len(svals):
        return 0.0
    return float(svals[r])


# --- spectral gap (top-two magnitude eigenvalues) ---
def spectral_gap(eigenvalues):
    mags = np.sort(np.abs(eigenvalues))[::-1]
    return float(mags[0] - mags[1]) if len(mags) > 1 else np.nan


# --- spurious eigenvalues vs reference ---


def spurious_ref(est, ref, delta):
    dist = np.abs(est[:, None] - ref[None, :])
    return int(np.sum(dist.min(axis=1) > delta))


def spurious_residual(eigenvalues, psi_X_val, psi_Y_val, delta, relative=True):
    r"""Data-driven spurious-eigenpair check (see paper Appendix C, Remark 4).

    Flags eigenpairs that fail the empirical consistency check
    :math:`\hat\psi_i(y_j) \approx \hat\lambda_i \hat\psi_i(x_j)` on a
    held-out validation set.

    Parameters
    ----------
    eigenvalues : ndarray, shape (r,)
        Estimated eigenvalues, same order as columns of psi_X_val/psi_Y_val.
    psi_X_val : ndarray, shape (n_val, r)
        Eigenfunctions evaluated at validation inputs x_j.
    psi_Y_val : ndarray, shape (n_val, r)
        Same eigenfunctions evaluated at outputs y_j.
    delta : float
        Threshold on the residual score.
    relative : bool
        If True, normalize residual by ||psi_X_val||.

    Returns
    -------
    n_spurious : int
    scores : ndarray, shape (r,)
    """
    eigenvalues = np.asarray(eigenvalues)
    n_val = psi_X_val.shape[0]

    resid = psi_Y_val - psi_X_val * eigenvalues[None, :]
    resid_norm = weighted_norm(resid) / np.sqrt(n_val)

    if relative:
        base_norm = weighted_norm(psi_X_val) / np.sqrt(n_val)
        scores = np.full_like(resid_norm, np.nan, dtype=float)

        ok = np.isfinite(base_norm) & (base_norm > 0)
        scores[ok] = resid_norm[ok] / base_norm[ok]
    else:
        scores = resid_norm

    n_spurious = int(np.sum(scores > delta))
    return n_spurious, scores


# --- compilation function for analysing spectral metrics ---
def analyse_spectrum(modes_records, trials_records, out_prefix):
    modes_df = pd.DataFrame(modes_records).copy()
    trials_df = pd.DataFrame(trials_records).copy()

    if "spectral_gap" not in modes_df.columns:
        raise ValueError(
            f"modes_df is missing 'spectral_gap'. Columns: {modes_df.columns.tolist()}"
        )

    summary = modes_df.groupby(
        ["kernel", "kind", "method", "eigenfunction_id"], as_index=False
    ).agg(
        n=("spectral_bias", "size"),
        bias_mean=("spectral_bias", "mean"),
        bias_std=("spectral_bias", "std"),
        dist_mean=("metric_distortion", "mean"),
        trunc_mean=("truncation", "mean"),
        spurious_mean=("residual_spurious_score", "mean"),
        spurious_std=("residual_spurious_score", "std"),
    )

    rows = []
    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        gg = g[["spectral_bias", "spectral_gap"]].dropna()
        corr = gg["spectral_bias"].corr(gg["spectral_gap"]) if len(gg) > 1 else np.nan
        rows.append(
            {
                "kernel": kernel,
                "kind": kind,
                "method": method,
                "bias_gap_corr": corr,
            }
        )
    corr_df = pd.DataFrame(rows)

    summary.to_csv(f"{out_prefix}_summary.csv", index=False)
    modes_df.to_csv(f"{out_prefix}_metrics.csv", index=False)
    trials_df.to_csv(f"{out_prefix}_trials.csv", index=False)
    corr_df.to_csv(f"{out_prefix}_corr.csv", index=False)

    # ------------------------------------------------------------
    # Scatter diagnostics with compact, deduplicated legends
    # ------------------------------------------------------------

    def _short_kernel_label(x, max_chars=20):
        s = str(x)
        return s if len(s) <= max_chars else s[: max_chars - 1] + "…"

    def _build_group_label(kernel, kind, method, include_kernel=False):
        """
        Keep legend labels compact.
        By default, legend shows only kind + method.
        Set include_kernel=True only if the number of groups is small.
        """
        if include_kernel:
            return f"{_short_kernel_label(kernel)}, {kind} / {method}"
        return f"{kind} / {method}"

    def _dedup_legend(ax, title=None, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8):
        handles, labels = ax.get_legend_handles_labels()
        seen = {}
        for h, l in zip(handles, labels):
            if l and l != "_nolegend_" and l not in seen:
                seen[l] = h

        if seen:
            ax.legend(
                seen.values(),
                seen.keys(),
                frameon=False,
                fontsize=fontsize,
                title=title,
                loc=loc,
                bbox_to_anchor=bbox_to_anchor,
                borderaxespad=0.0,
            )

    # -----------------------------
    # Figure 1: spectral bias vs spectral gap
    # -----------------------------
    fig1, ax = plt.subplots(figsize=(8.5, 4.8))

    # Set this to True only when there are very few groups
    include_kernel_in_legend = True

    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        g = g[np.isfinite(g["spectral_bias"]) & np.isfinite(g["spectral_gap"])].copy()
        if g.empty:
            continue

        label = _build_group_label(
            kernel=kernel,
            kind=kind,
            method=method,
            include_kernel=include_kernel_in_legend,
        )

        ax.scatter(
            g["spectral_bias"],
            g["spectral_gap"],
            s=20,
            alpha=0.7,
            label=label,
        )

    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Spectral gap")
    ax.set_title("Spectral bias vs Spectral gap")
    ax.grid(alpha=0.25)

    _dedup_legend(
        ax,
        title="Condition / method",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=8,
    )

    fig1.tight_layout()
    fig1.savefig(f"{out_prefix}_gap_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig1)

    # -----------------------------
    # Figure 2: spectral bias vs residual spurious score
    # -----------------------------
    fig2, ax = plt.subplots(figsize=(8.5, 4.8))

    for (kernel, kind, method), g in modes_df.groupby(["kernel", "kind", "method"]):
        g = g[np.isfinite(g["spectral_bias"]) & np.isfinite(g["residual_spurious_score"])].copy()
        if g.empty:
            continue

        label = _build_group_label(
            kernel=kernel,
            kind=kind,
            method=method,
            include_kernel=include_kernel_in_legend,
        )

        ax.scatter(
            g["spectral_bias"],
            g["residual_spurious_score"],
            s=20,
            alpha=0.7,
            label=label,
        )

    ax.set_xlabel("Spectral bias")
    ax.set_ylabel("Residual spurious score")
    ax.set_title("Spectral bias vs Residual spurious score")
    ax.grid(alpha=0.25)

    _dedup_legend(
        ax,
        title="Condition / method",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        fontsize=8,
    )

    fig2.tight_layout()
    fig2.savefig(f"{out_prefix}_spurious_scatter.png", dpi=200, bbox_inches="tight")
    plt.close(fig2)

    return modes_df, trials_df, summary, corr_df, fig1, fig2


In [ ]:
from collections.abc import Mapping


def _normalise_series(s, method="zscore", larger_is_better=False):
    x = pd.to_numeric(s, errors="coerce").astype(float)
    if larger_is_better:
        x = -x

    x = x.replace([np.inf, -np.inf], np.nan)
    m = x.notna()

    out = pd.Series(np.nan, index=x.index, dtype=float)

    if m.sum() == 0:
        return out.fillna(0.0)

    vals = x[m].to_numpy(dtype=float)

    if method == "zscore":
        mu = np.nanmean(vals)
        sd = np.nanstd(vals)
        if not np.isfinite(sd) or sd == 0:
            out.loc[m] = 0.0
        else:
            out.loc[m] = (vals - mu) / sd

    elif method == "minmax":
        lo = np.nanmin(vals)
        hi = np.nanmax(vals)
        if not np.isfinite(hi - lo) or hi == lo:
            out.loc[m] = 0.0
        else:
            out.loc[m] = (vals - lo) / (hi - lo)

    elif method == "rank":
        out.loc[m] = pd.Series(vals).rank(method="average").to_numpy(dtype=float)
        out.loc[m] = out.loc[m].max() - out.loc[m]
        denom = out.loc[m].max()
        if denom > 0:
            out.loc[m] = out.loc[m] / denom
        else:
            out.loc[m] = 0.0
    else:
        raise ValueError(f"Unknown normalise method: {method}")

    return out.fillna(0.0)


def kernel_spectral_score(
    summary,
    trials_df=None,
    *,
    group_cols=("kernel", "kind", "method"),
    selected_modes=None,
    mode_weights=None,
    normalise="zscore",
    metric_weights=None,
    hard_constraints=None,
    use_trial_metrics=True,
):
    """
    Second-stage kernel scoring built on top of analyse_spectrum outputs.

    Parameters
    ----------
    summary : pd.DataFrame
        Output `summary` from analyse_spectrum(...), with columns:
        ['kernel', 'kind', 'method', 'eigenfunction_id',
         'n', 'bias_mean', 'bias_std', 'dist_mean',
         'trunc_mean', 'spurious_mean', 'spurious_std']
    trials_df : pd.DataFrame or None
        Output `trials_df` from analyse_spectrum(...), with columns:
        ['kind', 'method', 'trial', 'spurious_ref_count',
         'spurious_residual_count', 'spectral_gap', 'rank']
        Optional, but recommended.
    group_cols : tuple[str, ...]
        Kernel identity columns.
    selected_modes : sequence[int] or None
        Which eigenfunction_id values to use. None means all.
    mode_weights : Mapping[int, float] or None
        Per-mode weights. If None, uses equal weights over selected modes.
    normalise : {"zscore", "minmax", "rank"}
        Normalisation used across candidate kernels.
    metric_weights : Mapping[str, float] or None
        Weights for the normalised metrics.
    hard_constraints : Mapping[str, float] or None
        Optional admissibility thresholds, e.g.
        {"max_spurious_ref_count": 2, "max_dist_mean": 20, "min_spectral_gap": 0.02}
    use_trial_metrics : bool
        If True and trials_df is provided, include trial-level aggregates.

    Returns
    -------
    mode_agg_df : pd.DataFrame
        Per-kernel aggregated raw metrics before normalisation.
    kernel_scores_df : pd.DataFrame
        Per-kernel normalised metrics, admissibility flags, composite score, rank.
    """

    summary = summary.copy()

    if selected_modes is not None:
        summary = summary[summary["eigenfunction_id"].isin(selected_modes)].copy()

    if summary.empty:
        raise ValueError("No rows remain in summary after filtering selected_modes.")

    if mode_weights is None:
        present_modes = sorted(summary["eigenfunction_id"].unique())
        mode_weights = {m: 1.0 for m in present_modes}
    elif isinstance(mode_weights, Mapping):
        mode_weights = dict(mode_weights)
    else:
        mode_weights = {m: w for m, w in mode_weights}

    summary["mode_weight"] = summary["eigenfunction_id"].map(mode_weights).fillna(0.0)
    if (summary["mode_weight"] < 0).any():
        raise ValueError("mode_weights must be nonnegative.")

    def _wavg(g, col):
        w = g["mode_weight"].to_numpy(dtype=float)
        x = g[col].to_numpy(dtype=float)
        if np.all(w == 0) or np.all(~np.isfinite(x)):
            return np.nan
        return float(np.average(x, weights=w))

    mode_agg_df = (
        summary.groupby(list(group_cols), as_index=False)
        .apply(
            lambda g: pd.Series(
                {
                    "n_modes_used": int(g["eigenfunction_id"].nunique()),
                    "weight_sum": float(g["mode_weight"].sum()),
                    "agg_bias_mean": _wavg(g, "bias_mean"),
                    "agg_bias_std": _wavg(g, "bias_std"),
                    "agg_dist_mean": _wavg(g, "dist_mean"),
                    "agg_trunc_mean": _wavg(g, "trunc_mean"),
                    "agg_spurious_mean": _wavg(g, "spurious_mean"),
                    "agg_spurious_std": _wavg(g, "spurious_std"),
                }
            )
        )
        .reset_index(drop=True)
    )

    if use_trial_metrics and trials_df is not None:
        trials_df = trials_df.copy()
        trial_group_cols = [c for c in group_cols if c in trials_df.columns]

        trial_agg = trials_df.groupby(trial_group_cols, as_index=False).agg(
            mean_spurious_ref_count=("spurious_ref_count", "mean"),
            mean_spurious_residual_count=("spurious_residual_count", "mean"),
            mean_spectral_gap=("spectral_gap", "mean"),
            std_spectral_gap=("spectral_gap", "std"),
            mean_rank=("rank", "mean"),
        )

        mode_agg_df = mode_agg_df.merge(trial_agg, on=trial_group_cols, how="left")

    if metric_weights is None:
        metric_weights = {
            "agg_bias_mean": 1.0,
            "agg_dist_mean": 1.0,
            "agg_spurious_mean": 1.0,
            "agg_trunc_mean": 0.5,
            "agg_bias_std": 0.25,
            "agg_spurious_std": 0.25,
            "mean_spurious_ref_count": 0.75,
            "mean_spurious_residual_count": 0.75,
            "mean_spectral_gap": 0.75,
            "std_spectral_gap": 0.25,
        }

    larger_is_better = {"mean_spectral_gap": True}

    kernel_scores_df = mode_agg_df.copy()

    used_metrics = []
    for metric, weight in metric_weights.items():
        if metric not in kernel_scores_df.columns or weight == 0:
            continue
        kernel_scores_df[f"{metric}_norm"] = _normalise_series(
            kernel_scores_df[metric],
            method=normalise,
            larger_is_better=larger_is_better.get(metric, False),
        )
        used_metrics.append(metric)

    score = np.zeros(len(kernel_scores_df), dtype=float)
    for metric, weight in metric_weights.items():
        norm_col = f"{metric}_norm"
        if norm_col in kernel_scores_df.columns and weight != 0:
            score += weight * kernel_scores_df[norm_col].to_numpy(dtype=float)

    kernel_scores_df["composite_score"] = score
    kernel_scores_df["used_metrics"] = ", ".join(used_metrics)

    kernel_scores_df["admissible"] = True
    kernel_scores_df["constraint_violations"] = ""

    if hard_constraints:
        admissible = np.ones(len(kernel_scores_df), dtype=bool)
        viol = [[] for _ in range(len(kernel_scores_df))]

        def _mark(mask, label):
            nonlocal admissible
            if mask is not None:
                admissible &= ~mask
                for i in np.where(mask)[0]:
                    viol[i].append(label)

        if (
            "max_spurious_ref_count" in hard_constraints
            and "mean_spurious_ref_count" in kernel_scores_df.columns
        ):
            _mark(
                kernel_scores_df["mean_spurious_ref_count"]
                > hard_constraints["max_spurious_ref_count"],
                "spurious_ref",
            )

        if (
            "max_spurious_residual_count" in hard_constraints
            and "mean_spurious_residual_count" in kernel_scores_df.columns
        ):
            _mark(
                kernel_scores_df["mean_spurious_residual_count"]
                > hard_constraints["max_spurious_residual_count"],
                "spurious_residual",
            )

        if "max_dist_mean" in hard_constraints:
            _mark(
                kernel_scores_df["agg_dist_mean"] > hard_constraints["max_dist_mean"], "distortion"
            )

        if "max_bias_mean" in hard_constraints:
            _mark(kernel_scores_df["agg_bias_mean"] > hard_constraints["max_bias_mean"], "bias")

        if "max_trunc_mean" in hard_constraints:
            _mark(
                kernel_scores_df["agg_trunc_mean"] > hard_constraints["max_trunc_mean"],
                "truncation",
            )

        if (
            "min_spectral_gap" in hard_constraints
            and "mean_spectral_gap" in kernel_scores_df.columns
        ):
            _mark(
                kernel_scores_df["mean_spectral_gap"] < hard_constraints["min_spectral_gap"], "gap"
            )

        kernel_scores_df["admissible"] = admissible
        kernel_scores_df["constraint_violations"] = [",".join(v) for v in viol]

    kernel_scores_df = kernel_scores_df.sort_values(
        ["admissible", "composite_score"],
        ascending=[False, True],
    ).reset_index(drop=True)

    kernel_scores_df["rank"] = np.arange(1, len(kernel_scores_df) + 1)

    return mode_agg_df, kernel_scores_df


In [ ]:
def plot_kernel_rankings(
    kernel_scores,
    trials_df=None,
    prefix="Kernel ranking",
    score_col="composite_score",
    kernel_col="kernel",
    method_col="method",
    facet_col="kind",
    rank_col="rank",
    gap_col_candidates=("gap_mean", "spectral_gap", "gap"),
    figsize_scale=(6, 4),
    annotate_bars=True,
    annotate_scatter=False,
    max_label_chars=28,
    sort_facets=True,
    sort_methods=True,
    sort_kernels_by_score=True,
    color_by_facet=True,
):
    """
    General plotting utility for kernel ranking outputs.

    Parameters
    ----------
    kernel_scores : pd.DataFrame
        Must contain at least kernel_col, method_col, score_col.
        May also contain facet_col, rank_col, and a gap column.
    trials_df : pd.DataFrame or None
        Optional trial-level dataframe used to compute mean spectral gap
        if kernel_scores does not already contain one.
    prefix : str
        Figure title prefix.
    score_col, kernel_col, method_col, facet_col, rank_col : str
        Column names to use.
    gap_col_candidates : tuple[str]
        Candidate names for gap columns.
    figsize_scale : tuple[float, float]
        Base width, height per panel.
    annotate_bars : bool
        Add rank labels above bars.
    annotate_scatter : bool
        Add kernel labels on scatter points.
    max_label_chars : int
        Maximum kernel label length before truncation.
    sort_facets, sort_methods, sort_kernels_by_score : bool
        Sorting behavior.
    color_by_facet : bool
        In scatter plot, color points by facet values. If False, use one color.
    """

    plot_df = kernel_scores.copy()

    # ---------- Validation ----------
    required = [kernel_col, method_col, score_col]
    missing = [c for c in required if c not in plot_df.columns]
    if missing:
        raise ValueError(f"kernel_scores is missing required columns: {missing}")

    # ---------- Ensure facet column exists ----------
    if facet_col is None or facet_col not in plot_df.columns:
        facet_col = "_facet"
        plot_df[facet_col] = "All"

    # ---------- Compute rank if missing ----------
    if rank_col not in plot_df.columns:
        group_cols = [c for c in [facet_col, method_col] if c in plot_df.columns]
        plot_df[rank_col] = (
            plot_df.groupby(group_cols)[score_col].rank(method="first", ascending=True).astype(int)
        )

    # ---------- Resolve / attach gap column ----------
    resolved_gap_col = None
    for c in gap_col_candidates:
        if c in plot_df.columns:
            resolved_gap_col = c
            break

    if resolved_gap_col is None and trials_df is not None:
        trial_df = trials_df.copy()

        if facet_col not in trial_df.columns:
            trial_df[facet_col] = "All"

        trial_gap_col = None
        for c in gap_col_candidates:
            if c in trial_df.columns:
                trial_gap_col = c
                break

        if trial_gap_col is not None:
            candidate_group_cols = [kernel_col, facet_col, method_col]
            group_cols = [c for c in candidate_group_cols if c in trial_df.columns]
            gap_df = (
                trial_df.groupby(group_cols, as_index=False)[trial_gap_col]
                .mean()
                .rename(columns={trial_gap_col: "gap_mean"})
            )

            merge_cols = [
                c for c in candidate_group_cols if c in plot_df.columns and c in gap_df.columns
            ]
            plot_df = plot_df.merge(gap_df, on=merge_cols, how="left")
            resolved_gap_col = "gap_mean"

    # ---------- Label helpers ----------
    def short_label(x, max_chars=max_label_chars):
        s = str(x)
        return s if len(s) <= max_chars else s[: max_chars - 1] + "…"

    def _dedup_legend(ax, title=None, loc="upper left", bbox_to_anchor=(1.02, 1.0), fontsize=8):
        handles, labels = ax.get_legend_handles_labels()
        seen = {}
        for h, l in zip(handles, labels):
            if l and l != "_nolegend_" and l not in seen:
                seen[l] = h

        if seen:
            ax.legend(
                seen.values(),
                seen.keys(),
                frameon=False,
                fontsize=fontsize,
                title=title,
                loc=loc,
                bbox_to_anchor=bbox_to_anchor,
                borderaxespad=0.0,
            )

    plot_df["_kernel_label"] = plot_df[kernel_col].map(short_label)

    # ---------- Ordering ----------
    facets = list(plot_df[facet_col].dropna().unique())
    methods = list(plot_df[method_col].dropna().unique())

    if sort_facets:
        try:
            facets = sorted(facets)
        except Exception:
            pass

    if sort_methods:
        try:
            methods = sorted(methods)
        except Exception:
            pass

    # ---------- Figure 1: bar rankings ----------
    fig1, axes = plt.subplots(
        len(facets),
        len(methods),
        figsize=(figsize_scale[0] * len(methods), figsize_scale[1] * len(facets)),
        squeeze=False,
    )

    for i, facet_val in enumerate(facets):
        for j, method in enumerate(methods):
            ax = axes[i, j]
            g = plot_df[(plot_df[facet_col] == facet_val) & (plot_df[method_col] == method)].copy()
            g = g[np.isfinite(g[score_col])].copy()

            if sort_kernels_by_score:
                g = g.sort_values(score_col, ascending=True)
            elif rank_col in g.columns:
                g = g.sort_values(rank_col, ascending=True)

            if g.empty:
                ax.set_visible(False)
                continue

            x = np.arange(len(g))
            ax.bar(x, g[score_col], color="#6ebdc2", alpha=0.85)
            ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
            ax.set_xticks(x)
            ax.set_xticklabels(g["_kernel_label"], rotation=45, ha="right")
            ax.set_ylabel("Composite score")
            ax.set_title(f"{prefix}: {facet_col}={facet_val} / {method}")
            ax.grid(axis="y", alpha=0.25)

            if annotate_bars and rank_col in g.columns:
                for xi, (_, row) in zip(x, g.iterrows()):
                    ax.annotate(
                        f"#{int(row[rank_col])}",
                        xy=(xi, row[score_col]),
                        xytext=(0, 4),
                        textcoords="offset points",
                        ha="center",
                        va="bottom",
                        fontsize=8,
                        clip_on=True,
                    )

    fig1.subplots_adjust(bottom=0.25, hspace=0.35, wspace=0.25)
    fig1.savefig(f"{prefix}_ranking_score.png", dpi=200)
    plt.show()

    # ---------- Figure 2: score vs gap ----------
    if resolved_gap_col is not None and resolved_gap_col in plot_df.columns:
        fig2, axes = plt.subplots(
            1,
            len(methods),
            figsize=(figsize_scale[0] * len(methods), figsize_scale[1]),
            squeeze=False,
        )

        cmap = plt.get_cmap("tab10")
        facet_to_color = {facet: cmap(k % 10) for k, facet in enumerate(facets)}

        for j, method in enumerate(methods):
            ax = axes[0, j]

            for facet_val in facets:
                g = plot_df[
                    (plot_df[method_col] == method) & (plot_df[facet_col] == facet_val)
                ].copy()
                g = g[np.isfinite(g[score_col]) & np.isfinite(g[resolved_gap_col])].copy()

                if g.empty or resolved_gap_col not in g.columns:
                    continue

                color = facet_to_color[facet_val] if color_by_facet else None

                ax.scatter(
                    g[resolved_gap_col],
                    g[score_col],
                    s=60,
                    alpha=0.8,
                    label=str(facet_val),
                    color=color,
                )

                if annotate_scatter:
                    for _, row in g.iterrows():
                        ax.annotate(
                            row["_kernel_label"],
                            (row[resolved_gap_col], row[score_col]),
                            xytext=(4, 4),
                            textcoords="offset points",
                            fontsize=8,
                        )

            ax.set_xlabel("Mean spectral gap")
            ax.set_ylabel("Composite score")
            ax.set_title(f"{prefix} score vs gap: {method}")
            ax.grid(alpha=0.25)

            _dedup_legend(
                ax,
                title="Condition / method",
                loc="upper left",
                bbox_to_anchor=(1.02, 1.0),
                fontsize=8,
            )
        fig2.savefig(f"{prefix}_score_gap.png", dpi=200, bbox_inches="tight")
        plt.show()
    else:
        print("No spectral gap column found or derivable; skipping score-vs-gap scatter.")

    return plot_df


In [ ]:
import re


def extract_tau(kernel):
    m = re.search(r"tau=(([0-9.eE+-]+))", str(kernel))
    return m.group(1) if m else str(kernel)


def extract_beta(kernel):
    m = re.search(r"beta=(([0-9.eE+-]+))", str(kernel))
    return m.group(1) if m else str(kernel)


In [ ]:
# ------------------------------------------------------------
# RQ4: finite graph/state-space kernel experiment
# True graph kernels on node identities
# ------------------------------------------------------------

# -----------------------------
# graph / Markov chain builders
# -----------------------------
def make_cycle_graph_transition(n_nodes):
    P = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        P[i, (i - 1) % n_nodes] = 0.5
        P[i, (i + 1) % n_nodes] = 0.5
    return P


def make_path_graph_transition(n_nodes):
    P = np.zeros((n_nodes, n_nodes), dtype=float)
    for i in range(n_nodes):
        nbrs = []
        if i > 0:
            nbrs.append(i - 1)
        if i < n_nodes - 1:
            nbrs.append(i + 1)
        for j in nbrs:
            P[i, j] = 1.0 / len(nbrs)
    return P


def graph_laplacian_from_P(P):
    A = 0.5 * (P + P.T)
    A = (A > 0).astype(float)
    np.fill_diagonal(A, 0.0)
    D = np.diag(A.sum(axis=1))
    return D - A


def stationary_dist(P, n_iter=5000):
    v = np.ones(P.shape[0]) / P.shape[0]
    for _ in range(n_iter):
        v = v @ P
    v = np.maximum(v, 0)
    v /= v.sum()
    return v


def simulate_markov_chain(P, n_steps, random_state, x0=0):
    rng = np.random.default_rng(random_state)
    states = np.empty(n_steps, dtype=int)
    states[0] = x0
    for t in range(n_steps - 1):
        states[t + 1] = rng.choice(P.shape[0], p=P[states[t]])
    return states


# -----------------------------
# graph kernels on node labels
# each input x is a 1D array carrying one integer node id
# -----------------------------
def delta_kernel():
    def kernel(x, y):
        i = int(np.asarray(x).ravel()[0])
        j = int(np.asarray(y).ravel()[0])
        return 1.0 if i == j else 0.0

    return kernel


def diffusion_kernel(L, tau):
    K = expm(-tau * L)

    def kernel(x, y):
        i = int(np.asarray(x).ravel()[0])
        j = int(np.asarray(y).ravel()[0])
        return float(K[i, j])

    return kernel


def resolvent_walk_kernel(P, beta):
    K = np.linalg.inv(np.eye(P.shape[0]) - beta * P)

    def kernel(x, y):
        i = int(np.asarray(x).ravel()[0])
        j = int(np.asarray(y).ravel()[0])
        return float(K[i, j])

    return kernel


# optional: exact feature map for delta kernel on finite states
def state_dataframe(states):
    return pd.DataFrame({"state": states.astype(float)})


## Delta Kernel

In [ ]:
# -----------------------------
# experiment setup
# -----------------------------

import itertools


graph_cases = {
    "cycle_12": make_cycle_graph_transition(12),
    "path_12": make_path_graph_transition(12),
}

n_train = 2000
n_val = 500
n_trials = 10
n_components = 5

trials_records_delta = []
modes_records_delta = []

for graph, method, reduced_rank in itertools.product(
    graph_cases.keys(),
    ["Principal Components (kDMD)", "Reduced Rank"],
    [False, True],
):
    if method == "Principal Components (kDMD)" and reduced_rank:
        continue
    if method == "Reduced Rank" and not reduced_rank:
        continue

    P = graph_cases[graph]
    L = graph_laplacian_from_P(P)
    vals_ref = np.linalg.eigvals(P)
    vals_ref = vals_ref[np.argsort(-np.abs(vals_ref))][:n_components]

    for trial in tqdm(range(n_trials), desc=f"RQ4 {graph} / {method}"):
        states_train = simulate_markov_chain(P, n_steps=n_train, random_state=1000 + trial, x0=0)
        states_val = simulate_markov_chain(P, n_steps=n_val, random_state=9000 + trial, x0=0)

        data = state_dataframe(states_train)
        data_val = state_dataframe(states_val)

        X_val = data_val.iloc[:-1].values
        Y_val = data_val.iloc[1:].values

        kernel = delta_kernel()
        model = KernelRidge(
            n_components=n_components,
            reduced_rank=reduced_rank,
            kernel=kernel,
            alpha=1e-10,
            random_state=trial,
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_ref(vals_hat, vals_ref, delta=0.15)

        vals_check, psi_X_val_raw = model.eig(eval_right_on=X_val)
        _, psi_Y_val_raw = model.eig(eval_right_on=Y_val)

        assert np.allclose(np.sort(np.abs(vals_check)), np.sort(np.abs(vals_hat))), (
            "eig() ordering changed between calls"
        )

        psi_X_val = psi_X_val_raw[:, sort_perm]
        psi_Y_val = psi_Y_val_raw[:, sort_perm]

        n_residual, spur_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=0.1, relative=True
        )

        fit_rank = model.rank_
        rho = (
            kdmd_truncation(C, fit_rank)
            if method == "Principal Components (kDMD)"
            else rrr_truncation(C, T, fit_rank)
        )

        trials_records_delta.append(
            {
                "kind": graph,
                "kernel": "delta",
                "method": method,
                "trial": trial,
                "spurious_ref_count": int(n_spur),
                "spurious_residual_count": int(n_residual),
                "spectral_gap": float(gap),
                "rank": fit_rank,
            }
        )

        n_modes = min(model.rank_, funcs_hat.shape[1])
        for j in range(n_modes):
            bias, distortion = spectral_bias(funcs_hat[:, j], C, rho)
            modes_records_delta.append(
                {
                    "kind": graph,
                    "kernel": "delta",
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "residual_spurious_score": float(spur_scores[j]),
                    "spectral_gap": float(gap),
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                    "gap": float(gap),
                }
            )

modes_df_delta, trials_df_delta, summary_delta, corr_df_delta, fig1_delta, fig2_delta = (
    analyse_spectrum(modes_records_delta, trials_records_delta, out_prefix="graph_delta")
)

mode_agg_delta, kernel_scores_delta = kernel_spectral_score(
    summary_delta,
    trials_df_delta,
    group_cols=("kernel", "kind", "method"),
    selected_modes=(1, 2, 3),
    mode_weights={1: 0.5, 2: 0.3, 3: 0.2},
    normalise="zscore",
    metric_weights={
        "agg_bias_mean": 1.0,
        "agg_dist_mean": 1.0,
        "agg_spurious_mean": 1.0,
        "agg_trunc_mean": 0.5,
        "agg_bias_std": 0.25,
        "agg_spurious_std": 0.25,
        "mean_spurious_ref_count": 0.5,
        "mean_spurious_residual_count": 1.0,
        "mean_spectral_gap": 0.5,
        "std_spectral_gap": 0.25,
    },
    hard_constraints={
        "max_spurious_residual_count": 5,
    },
)
mode_agg_delta.to_csv("graph_delta_modes.csv", index=False)
kernel_scores_delta.to_csv("graph_delta_scores.csv", index=False)


delta_scores_sorted = kernel_scores_delta.sort_values(
    ["kind", "method", "rank", "kernel"]
).reset_index(drop=True)

delta_scores_sorted.to_csv("graph_delta_ranking.csv", index=False)

plot_kernel_rankings(kernel_scores_delta, trials_df_delta, prefix="graph_delta")


## Diffusion kernel

In [ ]:
# -----------------------------
# experiment setup
# -----------------------------


graph_cases = {
    "cycle_12": make_cycle_graph_transition(12),
    "path_12": make_path_graph_transition(12),
}

taus = [1e-3, 2e-3, 5e-3, 1e-2, 2e-2, 5e-2, 1e-1, 2e-1, 5e-1, 1, 2, 5]

n_train = 2000
n_val = 500
n_trials = 10
n_components = 5

trials_records_diff = []
modes_records_diff = []

for graph, tau, method, reduced_rank in itertools.product(
    graph_cases.keys(),
    taus,
    ["Principal Components (kDMD)", "Reduced Rank"],
    [False, True],
):
    if method == "Principal Components (kDMD)" and reduced_rank:
        continue
    if method == "Reduced Rank" and not reduced_rank:
        continue

    P = graph_cases[graph]
    L = graph_laplacian_from_P(P)
    vals_ref = np.linalg.eigvals(P)
    vals_ref = vals_ref[np.argsort(-np.abs(vals_ref))][:n_components]

    kernel = diffusion_kernel(L, tau)

    for trial in tqdm(range(n_trials), desc=f"{graph} / tau={tau} / {method}"):
        states_train = simulate_markov_chain(P, n_steps=n_train, random_state=1000 + trial, x0=0)
        states_val = simulate_markov_chain(P, n_steps=n_val, random_state=9000 + trial, x0=0)

        data = state_dataframe(states_train)
        data_val = state_dataframe(states_val)

        X_val = data_val.iloc[:-1].values
        Y_val = data_val.iloc[1:].values

        model = KernelRidge(
            n_components=n_components,
            reduced_rank=reduced_rank,
            kernel=kernel,
            alpha=1e-10,
            random_state=trial,
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_ref(vals_hat, vals_ref, delta=0.15)

        vals_check, psi_X_val_raw = model.eig(eval_right_on=X_val)
        _, psi_Y_val_raw = model.eig(eval_right_on=Y_val)

        assert np.allclose(np.sort(np.abs(vals_check)), np.sort(np.abs(vals_hat))), (
            "eig() ordering changed between calls"
        )

        psi_X_val = psi_X_val_raw[:, sort_perm]
        psi_Y_val = psi_Y_val_raw[:, sort_perm]

        n_residual, spur_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=0.1, relative=True
        )

        fit_rank = model.rank_
        rho = (
            kdmd_truncation(C, fit_rank)
            if method == "Principal Components (kDMD)"
            else rrr_truncation(C, T, fit_rank)
        )

        trials_records_diff.append(
            {
                "kind": graph,
                "kernel": f"diffusion(tau={tau})",
                "method": method,
                "trial": trial,
                "spurious_ref_count": int(n_spur),
                "spurious_residual_count": int(n_residual),
                "spectral_gap": float(gap),
                "rank": fit_rank,
            }
        )

        n_modes = min(model.rank_, funcs_hat.shape[1])
        for j in range(n_modes):
            bias, distortion = spectral_bias(funcs_hat[:, j], C, rho)
            modes_records_diff.append(
                {
                    "kind": graph,
                    "kernel": f"diffusion(tau={tau})",
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "residual_spurious_score": float(spur_scores[j]),
                    "spectral_gap": float(gap),
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                    "gap": float(gap),
                }
            )

modes_df_diff, trials_df_diff, summary_diff, corr_df_diff, fig1_diff, fig2_diff = analyse_spectrum(
    modes_records_diff, trials_records_diff, out_prefix="diff_graph_kernel"
)

mode_agg_diff, kernel_scores_diff = kernel_spectral_score(
    summary_diff,
    trials_df_diff,
    group_cols=("kernel", "kind", "method"),
    selected_modes=(1, 2, 3),
    mode_weights={1: 0.5, 2: 0.3, 3: 0.2},
    normalise="zscore",
    metric_weights={
        "agg_bias_mean": 1.0,
        "agg_dist_mean": 1.0,
        "agg_spurious_mean": 1.0,
        "agg_trunc_mean": 0.5,
        "agg_bias_std": 0.25,
        "agg_spurious_std": 0.25,
        "mean_spurious_ref_count": 0.5,
        "mean_spurious_residual_count": 1.0,
        "mean_spectral_gap": 0.5,
        "std_spectral_gap": 0.25,
    },
    hard_constraints={
        "max_spurious_residual_count": 5,
    },
)
mode_agg_diff.to_csv("graph_diff_modes.csv", index=False)
kernel_scores_diff.to_csv("graph_diff_scores.csv", index=False)


diff_scores_sorted = kernel_scores_diff.sort_values(
    ["kind", "method", "rank", "kernel"]
).reset_index(drop=True)

diff_scores_sorted.to_csv("graph_diff_ranking.csv", index=False)

plot_kernel_rankings(kernel_scores_diff, trials_df_diff, prefix="graph_diff")


In [ ]:
# Plotting kernel effects
# -----------------------
plot_df = kernel_scores_diff.copy()

plot_df["tau"] = plot_df["kernel"].map(extract_tau)

# 5a. ranking bars by A and method
methods = ["Principal Components (kDMD)", "Reduced Rank"]

for graph in graph_cases:
    sub = plot_df.loc[plot_df["kind"].astype(str).eq(f"{graph}")].copy()
    sub["method"] = sub["method"].astype(str).str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    methods = ["Principal Components (kDMD)", "Reduced Rank"]

    for ax, method in zip(axes, methods):
        g = sub.loc[sub["method"].eq(method)].copy()
        g = g.sort_values(["rank", "tau"], ascending=[True, True]).reset_index(drop=True)
        x = np.arange(len(g))
        y = g["composite_score"].to_numpy()
        ypad = 0.03 * max(1.0, np.nanmax(np.abs(y)))

        ax.bar(x, y, color="#D667A0", width=0.75)
        ax.set_xticks(x)
        ax.set_xticklabels(g["tau"], rotation=90, ha="right")
        ax.set_title(method)
        ax.set_xlabel("Tau")
        ax.grid(axis="y", alpha=0.25)

        for i, (xi, yi, rk) in enumerate(zip(x, y, g["rank"])):
            va = "bottom" if yi >= 0 else "top"
            offset = ypad if yi >= 0 else -ypad
            dx = -0.10 if i % 2 == 0 else 0.10
            ax.text(
                xi + dx,
                yi + offset,
                f"#{int(rk)}",
                ha="center",
                va=va,
                fontsize=7,
                rotation=90,
            )

    axes[0].set_ylabel("Composite score")
    fig.suptitle(f"Graph diffusion kernel ranking ({graph})", fontsize=14)
    fig.tight_layout()
    fig.savefig(f"graph_diff_ranking_{graph}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print("Saved Graph diffusion output.")


## Resolvent walk kernel

In [ ]:
# -----------------------------
# experiment setup
# -----------------------------


graph_cases = {
    "cycle_12": make_cycle_graph_transition(12),
    "path_12": make_path_graph_transition(12),
}

betas = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1, 3e-1, 1, 3, 10]

n_train = 2000
n_val = 500
n_trials = 10
n_components = 5

trials_records_res = []
modes_records_res = []

for graph, beta, method, reduced_rank in itertools.product(
    graph_cases.keys(),
    betas,
    ["Principal Components (kDMD)", "Reduced Rank"],
    [False, True],
):
    if method == "Principal Components (kDMD)" and reduced_rank:
        continue
    if method == "Reduced Rank" and not reduced_rank:
        continue

    P = graph_cases[graph]
    L = graph_laplacian_from_P(P)
    vals_ref = np.linalg.eigvals(P)
    vals_ref = vals_ref[np.argsort(-np.abs(vals_ref))][:n_components]

    kernel = resolvent_walk_kernel(L, beta)

    for trial in tqdm(range(n_trials), desc=f"{graph} / beta={beta} / {method}"):
        states_train = simulate_markov_chain(P, n_steps=n_train, random_state=1000 + trial, x0=0)
        states_val = simulate_markov_chain(P, n_steps=n_val, random_state=9000 + trial, x0=0)

        data = state_dataframe(states_train)
        data_val = state_dataframe(states_val)

        X_val = data_val.iloc[:-1].values
        Y_val = data_val.iloc[1:].values

        model = KernelRidge(
            n_components=n_components,
            reduced_rank=reduced_rank,
            kernel=kernel,
            alpha=1e-10,
            random_state=trial,
        )
        model.fit(data)

        n = model.kernel_X_.shape[0]
        C = model.kernel_X_ / n
        T = model.kernel_YX_ / n

        vals_hat, funcs_hat = model.eig(eval_right_on=model.X_fit_[:-1])
        sort_perm = np.flip(np.argsort(np.abs(vals_hat)))
        vals_hat, funcs_hat = vals_hat[sort_perm], funcs_hat[:, sort_perm]

        gap = spectral_gap(vals_hat)
        n_spur = spurious_ref(vals_hat, vals_ref, delta=0.15)

        vals_check, psi_X_val_raw = model.eig(eval_right_on=X_val)
        _, psi_Y_val_raw = model.eig(eval_right_on=Y_val)

        assert np.allclose(np.sort(np.abs(vals_check)), np.sort(np.abs(vals_hat))), (
            "eig() ordering changed between calls"
        )

        psi_X_val = psi_X_val_raw[:, sort_perm]
        psi_Y_val = psi_Y_val_raw[:, sort_perm]

        n_residual, spur_scores = spurious_residual(
            vals_hat, psi_X_val, psi_Y_val, delta=0.1, relative=True
        )

        fit_rank = model.rank_
        rho = (
            kdmd_truncation(C, fit_rank)
            if method == "Principal Components (kDMD)"
            else rrr_truncation(C, T, fit_rank)
        )

        trials_records_res.append(
            {
                "kind": graph,
                "kernel": f"resolvent(beta={beta})",
                "method": method,
                "trial": trial,
                "spurious_ref_count": int(n_spur),
                "spurious_residual_count": int(n_residual),
                "spectral_gap": float(gap),
                "rank": fit_rank,
            }
        )

        n_modes = min(model.rank_, funcs_hat.shape[1])
        for j in range(n_modes):
            bias, distortion = spectral_bias(funcs_hat[:, j], C, rho)
            modes_records_res.append(
                {
                    "kind": graph,
                    "kernel": f"resolvent(beta={beta})",
                    "method": method,
                    "trial": trial,
                    "eigenfunction_id": j + 1,
                    "spectral_bias": float(bias),
                    "metric_distortion": float(distortion),
                    "truncation": float(rho),
                    "residual_spurious_score": float(spur_scores[j]),
                    "spectral_gap": float(gap),
                    "est_eig_real": float(np.real(vals_hat[j])),
                    "est_eig_imag": float(np.imag(vals_hat[j])),
                    "gap": float(gap),
                }
            )

modes_df_res, trials_df_res, summary_res, corr_df_res, fig1_res, fig2_res = analyse_spectrum(
    modes_records_res, trials_records_res, out_prefix="res_graph_kernel"
)

mode_agg_res, kernel_scores_res = kernel_spectral_score(
    summary_res,
    trials_df_res,
    group_cols=("kernel", "kind", "method"),
    selected_modes=(1, 2, 3),
    mode_weights={1: 0.5, 2: 0.3, 3: 0.2},
    normalise="zscore",
    metric_weights={
        "agg_bias_mean": 1.0,
        "agg_dist_mean": 1.0,
        "agg_spurious_mean": 1.0,
        "agg_trunc_mean": 0.5,
        "agg_bias_std": 0.25,
        "agg_spurious_std": 0.25,
        "mean_spurious_ref_count": 0.5,
        "mean_spurious_residual_count": 1.0,
        "mean_spectral_gap": 0.5,
        "std_spectral_gap": 0.25,
    },
    hard_constraints={
        "max_spurious_residual_count": 5,
    },
)
mode_agg_res.to_csv("graph_res_modes.csv", index=False)
kernel_scores_res.to_csv("graph_res_scores.csv", index=False)


res_scores_sorted = kernel_scores_res.sort_values(["kind", "method", "rank", "kernel"]).reset_index(
    drop=True
)

res_scores_sorted.to_csv("graph_res_ranking.csv", index=False)

plot_kernel_rankings(kernel_scores_res, trials_df_res, prefix="graph_res")


In [ ]:
# Plotting kernel effects
# -----------------------
plot_df = kernel_scores_res.copy()

plot_df["beta"] = plot_df["kernel"].map(extract_beta)

# 5a. ranking bars by A and method
methods = ["Principal Components (kDMD)", "Reduced Rank"]

for graph in graph_cases:
    sub = plot_df.loc[plot_df["kind"].astype(str).eq(f"{graph}")].copy()
    sub["method"] = sub["method"].astype(str).str.strip()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
    methods = ["Principal Components (kDMD)", "Reduced Rank"]

    for ax, method in zip(axes, methods):
        g = sub.loc[sub["method"].eq(method)].copy()
        g = g.sort_values(["rank", "beta"], ascending=[True, True]).reset_index(drop=True)
        x = np.arange(len(g))
        y = g["composite_score"].to_numpy()
        ypad = 0.03 * max(1.0, np.nanmax(np.abs(y)))

        ax.bar(x, y, color="#D667A0", width=0.75)
        ax.set_xticks(x)
        ax.set_xticklabels(g["beta"], rotation=90, ha="right")
        ax.set_title(method)
        ax.set_xlabel("Beta")
        ax.grid(axis="y", alpha=0.25)

        for i, (xi, yi, rk) in enumerate(zip(x, y, g["rank"])):
            va = "bottom" if yi >= 0 else "top"
            offset = ypad if yi >= 0 else -ypad
            dx = -0.10 if i % 2 == 0 else 0.10
            ax.text(
                xi + dx,
                yi + offset,
                f"#{int(rk)}",
                ha="center",
                va=va,
                fontsize=7,
                rotation=90,
            )

    axes[0].set_ylabel("Composite score")
    fig.suptitle(f"Graph resolvent walk kernel ranking ({graph})", fontsize=14)
    fig.tight_layout()
    fig.savefig(f"graph_res_ranking_{graph}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

print("Saved Graph resolvent walk output.")
